In [27]:
import pandas as pd
import numpy as np

import nltk, re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


In [2]:
df = pd.read_csv('src/fake_news_dataset.csv')

In [32]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)
X_tfidf = tfidf.fit_transform(df['text'])
y = df['label'].map({'fake': 1, 'real': 0}).values

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

In [34]:
clf_lr = LogisticRegression(
    max_iter=200,
    C=2.0,
    n_jobs=-1
)
clf_lr.fit(X_train, y_train)
pred = clf_lr.predict(X_test)

In [35]:
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
prec = precision_score(y_test, pred)
recall = recall_score(y_test, pred)

print(f'acc = {acc}\nf1 = {f1}\nprecision_score = {prec}\nrecall_score = {recall}')

acc = 0.5035
f1 = 0.5165530671859786
precision_score = 0.5059608965188365
recall_score = 0.5275982098458478


In [36]:
clf_svm = LinearSVC()
clf_svm.fit(X_train, y_train)
pred = clf_svm.predict(X_test)

In [37]:
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
prec = precision_score(y_test, pred)
recall = recall_score(y_test, pred)

print(f'acc = {acc}\nf1 = {f1}\nprecision_score = {prec}\nrecall_score = {recall}')

acc = 0.5045
f1 = 0.5146914789422136
precision_score = 0.5069946936806561
recall_score = 0.5226255594231726


In [38]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

In [39]:
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
prec = precision_score(y_test, pred)
recall = recall_score(y_test, pred)

print(f'acc = {acc}\nf1 = {f1}\nprecision_score = {prec}\nrecall_score = {recall}')

acc = 0.512
f1 = 0.5601622352410996
precision_score = 0.5121549237742068
recall_score = 0.618100447538538


In [17]:
stop_words = set(stopwords.words('english'))
stop_words.update(['mr', 'else'])
lemmatizer = WordNetLemmatizer()

def preprocess(text, reg=r'[^a-zA-Z\s]'):
    text = re.sub(reg, '', text.lower())
    tokens = nltk.word_tokenize(text)
    return [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]

In [18]:
df['tokens'] = df['text'].apply(preprocess)

In [14]:
labels = df['label'].tolist()

feature_names = np.array(tfidf.get_feature_names_out())

def top_tfidf_words(label_value, top_n=20):
    idx = np.where(np.array(labels) == label_value)[0]
    class_tfidf = X_tfidf[idx].mean(axis=0).A1
    top_idx = np.argsort(class_tfidf)[::-1][:top_n]
    return pd.DataFrame({
        'term': feature_names[top_idx],
        'mean_tfidf': class_tfidf[top_idx]
    })

top_fake = top_tfidf_words('fake')
top_real = top_tfidf_words('real')

In [19]:
fake_terms = top_fake['term'].tolist()
real_terms = top_real['term'].tolist()

fake_set = set(fake_terms)
real_set = set(real_terms)

def has_fake_tfidf(tokens):
    return int(any(t in fake_set for t in tokens))

def has_real_tfidf(tokens):
    return int(any(t in real_set for t in tokens))

df['has_fake_tfidf'] = df['tokens'].apply(has_fake_tfidf)
df['has_real_tfidf'] = df['tokens'].apply(has_real_tfidf)

In [21]:
def tfidf_counts(tokens):
    c = Counter(tokens)
    fake_count = sum(c[w] for w in fake_set)
    real_count = sum(c[w] for w in real_set)
    return pd.Series({'fake_tfidf_count': fake_count,
                      'real_tfidf_count': real_count})

df[['fake_tfidf_count', 'real_tfidf_count']] = df['tokens'].apply(tfidf_counts)

df['len_tokens'] = df['tokens'].apply(len)
df['fake_tfidf_frac'] = df['fake_tfidf_count'] / df['len_tokens'].clip(lower=1)
df['real_tfidf_frac'] = df['real_tfidf_count'] / df['len_tokens'].clip(lower=1)

In [42]:
e_cols = ['has_fake_tfidf', 'has_real_tfidf',
              'fake_tfidf_frac', 'real_tfidf_frac', 'len_tokens']
e_X = df[e_cols].to_numpy().astype(float)

In [24]:
X = hstack([X_tfidf, e_X])

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [28]:
clf = LogisticRegression(
    max_iter=200,
    C=2.0,
    n_jobs=-1
)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

In [29]:
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
prec = precision_score(y_test, pred)
recall = recall_score(y_test, pred)

print(f'acc = {acc}\nf1 = {f1}\nprecision_score = {prec}\nrecall_score = {recall}')

acc = 0.507
f1 = 0.5201946472019465
precision_score = 0.5092901381610291
recall_score = 0.5315763301839881
